Run these examples from the complete `tcia_utils_v4` folder; see [setup and preview-release instructions](README.md).

# Non-DICOM availability and annotation signals

This notebook inspects public asset summaries and DICOM annotation download signals. An asset may be a package or transformed preview. An annotation signal does not prove which source image it annotates. Live service routes may be unavailable even when readiness is green; that is reported explicitly.

In [ ]:
from importlib.metadata import version
from _support import MODE, SHORT_TITLE, DOI, make_client, checked
print("SYNTHETIC FIXTURE MODE — results below are examples, not live TCIA evidence" if MODE == "fixture"
      else "LIVE METADATA MODE — bounded calls to the configured TCIA service")
print("tcia_utils", version("tcia_utils"))

In [ ]:
with make_client() as tcia:
    assets = checked("public non-DICOM assets", lambda: tcia.search_public_assets(short_titles=[SHORT_TITLE], limit=1))
    if assets:
        for row in assets.items:
            print("asset", row.get("asset_id"), "format", row.get("file_format"),
                  "grain", row.get("asset_granularity"), "inventory", row.get("inventory_status"))
        print("scope", assets.response.get("scope"), "completeness", assets.completeness.state)
    annotations = checked("annotation downloads", lambda: tcia.search_annotation_downloads(short_titles=[SHORT_TITLE], limit=3))
    if annotations:
        print("annotation downloads", [(row.get("download_id"), row.get("access_level"))
                                       for row in annotations.data["downloads"]])
        print("relationship limit", annotations.data["scope"])
        print("bounded completeness", annotations.completeness.state)

Public non-DICOM inventory can include narrowly reviewed IDC-missing DICOM exceptions; it is not a general DICOM series catalog. If a route fails, retain the unavailable state rather than treating it as zero matching assets. No file or image is fetched.